In [1]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

In [2]:
EXPRESSION_PATH = Path(
    "../data/raw/GSE226105_rLog-Normalized_counts.csv"
)

ROBUSTNESS_DIR = Path(
    "../data/interim/robustness_checks"
)

ROBUSTNESS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
expression_data = pd.read_csv(
    EXPRESSION_PATH
)

if "gene_id" not in expression_data.columns:
    expression_data = expression_data.rename(
        columns={
            "Unnamed: 0": "gene_id"
        }
    )

expression_data.shape

(28775, 58)

In [4]:
expression = (
    expression_data
    .set_index("gene_id")
)

informative_expression = expression.loc[
    (expression != 0).any(axis=1)
]

informative_expression.shape

(26118, 57)

In [5]:
control_columns = [
    column
    for column in informative_expression.columns
    if column.startswith("CONT")
]

control_columns

['CONT 1', 'CONT 2', 'CONT 3']

In [6]:
control_pairwise_differences = []

for sample_a, sample_b in combinations(
    control_columns,
    2
):
    differences = (
        informative_expression[sample_a]
        - informative_expression[sample_b]
    ).abs()

    control_pairwise_differences.append(
        differences
    )

control_pairwise_differences = pd.concat(
    control_pairwise_differences,
    ignore_index=True
)

len(control_pairwise_differences)

78354

In [7]:
response_floors = (
    control_pairwise_differences
    .quantile(
        [
            0.90,
            0.95,
            0.99,
        ]
    )
)

response_floors

0.90    0.272532
0.95    0.342263
0.99    0.554173
dtype: float64

In [8]:
import re

In [9]:
condition_map = {
    "COLD HL": "cold_high_light",
    "COLD": "cold",
    "T": "heat_drought_high_light",
    "HEAT HL": "heat_high_light",
    "HEAT DROUGHT": "heat_drought",
    "HEAT": "heat",
    "DROUGHT": "drought",
    "Humidity": "high_humidity",
    "HL": "high_light",
}

In [10]:
manifest_rows = []

for column in informative_expression.columns:

    if column.startswith("CONT"):

        replicate = int(
            re.search(r"(\d+)$", column).group(1)
        )

        manifest_rows.append(
            {
                "matrix_column": column,
                "state": "control",
                "condition": "control",
                "replicate": replicate,
            }
        )

        continue

    if column.startswith("endpoint_"):
        state = "endpoint"

    elif column.startswith("recovery_"):
        state = "recovery"

    else:
        raise ValueError(
            f"Unrecognized sample column: {column}"
        )

    sample_body = column.split("_", 1)[1].strip()

    match = re.match(
        r"^(.*?)(?:\s+)?(\d+)$",
        sample_body
    )

    if match is None:
        raise ValueError(
            f"Could not parse sample: {column}"
        )

    raw_condition = match.group(1).strip()
    replicate = int(match.group(2))

    manifest_rows.append(
        {
            "matrix_column": column,
            "state": state,
            "condition": condition_map[raw_condition],
            "replicate": replicate,
        }
    )

sample_manifest = pd.DataFrame(
    manifest_rows
)

sample_manifest.head()

,matrix_column,state,condition,replicate
0,endpoint_COLD HL 1,endpoint,cold_high_light,1
1,endpoint_COLD HL 2,endpoint,cold_high_light,2
2,endpoint_COLD HL 3,endpoint,cold_high_light,3
3,endpoint_COLD 1,endpoint,cold,1
4,endpoint_COLD 2,endpoint,cold,2


In [11]:
sample_manifest.groupby(
    [
        "state",
        "condition",
    ]
).size()

state     condition              
control   control                    3
endpoint  cold                       3
          cold_high_light            3
          drought                    3
          heat                       3
          heat_drought               3
          heat_drought_high_light    3
          heat_high_light            3
          high_humidity              3
          high_light                 3
recovery  cold                       3
          cold_high_light            3
          drought                    3
          heat                       3
          heat_drought               3
          heat_drought_high_light    3
          heat_high_light            3
          high_humidity              3
          high_light                 3
dtype: int64

In [12]:
condition_order = [
    "cold",
    "cold_high_light",
    "drought",
    "heat",
    "heat_drought",
    "heat_drought_high_light",
    "heat_high_light",
    "high_humidity",
    "high_light",
]

In [13]:
control_columns = sample_manifest.loc[
    sample_manifest["state"] == "control",
    "matrix_column"
].tolist()

control_mean = (
    informative_expression[
        control_columns
    ]
    .mean(axis=1)
)

In [14]:
endpoint_means = {}

for condition in condition_order:

    columns = sample_manifest.loc[
        (sample_manifest["state"] == "endpoint")
        & (sample_manifest["condition"] == condition),
        "matrix_column"
    ].tolist()

    endpoint_means[condition] = (
        informative_expression[
            columns
        ]
        .mean(axis=1)
    )

endpoint_means = pd.DataFrame(
    endpoint_means
)

In [15]:
recovery_means = {}

for condition in condition_order:

    columns = sample_manifest.loc[
        (sample_manifest["state"] == "recovery")
        & (sample_manifest["condition"] == condition),
        "matrix_column"
    ].tolist()

    recovery_means[condition] = (
        informative_expression[
            columns
        ]
        .mean(axis=1)
    )

recovery_means = pd.DataFrame(
    recovery_means
)

In [16]:
endpoint_shift = (
    endpoint_means
    .sub(
        control_mean,
        axis=0
    )
)

recovery_shift = (
    recovery_means
    .sub(
        control_mean,
        axis=0
    )
)

In [17]:
endpoint_shift.shape

(26118, 9)

In [18]:
recovery_shift.shape

(26118, 9)

In [19]:
sample_manifest.groupby("state").size()

state
control      3
endpoint    27
recovery    27
dtype: int64

In [20]:
def classify_recovery_profiles(
    endpoint_shift,
    recovery_shift,
    response_floor,
):
    categories = pd.DataFrame(
        pd.NA,
        index=endpoint_shift.index,
        columns=endpoint_shift.columns,
        dtype="object",
    )

    for condition in endpoint_shift.columns:

        endpoint = endpoint_shift[condition]
        recovery = recovery_shift[condition]

        endpoint_eligible = (
            endpoint.abs() >= response_floor
        )

        recovery_residual = (
            recovery.abs() >= response_floor
        )

        same_direction = (
            endpoint * recovery > 0
        )

        opposite_direction = (
            endpoint * recovery < 0
        )

        categories.loc[
            endpoint_eligible
            & ~recovery_residual,
            condition,
        ] = "recovered"

        categories.loc[
            endpoint_eligible
            & recovery_residual
            & same_direction,
            condition,
        ] = "persistent_same_direction"

        categories.loc[
            endpoint_eligible
            & recovery_residual
            & opposite_direction,
            condition,
        ] = "substantial_reversal"

    return categories

In [21]:
def summarize_recovery_categories(
    recovery_categories,
):
    rows = []

    for condition in recovery_categories.columns:

        values = recovery_categories[condition]

        rows.append(
            {
                "condition": condition,
                "endpoint_eligible": (
                    values.notna().sum()
                ),
                "recovered": (
                    values.eq("recovered").sum()
                ),
                "persistent_same_direction": (
                    values.eq(
                        "persistent_same_direction"
                    ).sum()
                ),
                "substantial_reversal": (
                    values.eq(
                        "substantial_reversal"
                    ).sum()
                ),
            }
        )

    return pd.DataFrame(rows)

In [22]:
original_floor = response_floors.loc[0.95]

recovery_categories_95 = (
    classify_recovery_profiles(
        endpoint_shift=endpoint_shift,
        recovery_shift=recovery_shift,
        response_floor=original_floor,
    )
)

recovery_summary_95 = (
    summarize_recovery_categories(
        recovery_categories_95
    )
)

recovery_summary_95

,condition,endpoint_eligible,recovered,persistent_same_direction,substantial_reversal
0,cold,8915,8838,71,6
1,cold_high_light,12210,11677,187,346
2,drought,11942,10613,1240,89
3,heat,1310,1226,70,14
4,heat_drought,14088,13690,344,54
5,heat_drought_high_light,8609,8323,255,31
6,heat_high_light,814,729,72,13
7,high_humidity,1712,1661,28,23
8,high_light,1950,1154,779,17


In [23]:
recovery_summary_95["category_total"] = (
    recovery_summary_95["recovered"]
    + recovery_summary_95[
        "persistent_same_direction"
    ]
    + recovery_summary_95[
        "substantial_reversal"
    ]
)

recovery_summary_95[
    "categories_sum_to_endpoint_eligible"
] = (
    recovery_summary_95["category_total"]
    == recovery_summary_95["endpoint_eligible"]
)

recovery_summary_95

,condition,endpoint_eligible,recovered,persistent_same_direction,substantial_reversal,category_total,categories_sum_to_endpoint_eligible
0,cold,8915,8838,71,6,8915,True
1,cold_high_light,12210,11677,187,346,12210,True
2,drought,11942,10613,1240,89,11942,True
3,heat,1310,1226,70,14,1310,True
4,heat_drought,14088,13690,344,54,14088,True
5,heat_drought_high_light,8609,8323,255,31,8609,True
6,heat_high_light,814,729,72,13,814,True
7,high_humidity,1712,1661,28,23,1712,True
8,high_light,1950,1154,779,17,1950,True


In [24]:
floor_specs = {
    "p90": response_floors.loc[0.90],
    "p95": response_floors.loc[0.95],
    "p99": response_floors.loc[0.99],
}

recovery_categories_by_floor = {}
robustness_summary_tables = []

for floor_label, response_floor in floor_specs.items():

    categories = classify_recovery_profiles(
        endpoint_shift=endpoint_shift,
        recovery_shift=recovery_shift,
        response_floor=response_floor,
    )

    summary = summarize_recovery_categories(
        categories
    )

    summary["floor_label"] = floor_label
    summary["response_floor"] = response_floor

    summary["persistent_fraction_of_responders"] = (
        summary["persistent_same_direction"]
        / summary["endpoint_eligible"]
    )

    summary["recovered_fraction_of_responders"] = (
        summary["recovered"]
        / summary["endpoint_eligible"]
    )

    summary["reversal_fraction_of_responders"] = (
        summary["substantial_reversal"]
        / summary["endpoint_eligible"]
    )

    recovery_categories_by_floor[
        floor_label
    ] = categories

    robustness_summary_tables.append(
        summary
    )

response_floor_sensitivity = pd.concat(
    robustness_summary_tables,
    ignore_index=True,
)

response_floor_sensitivity

,condition,endpoint_eligible,recovered,persistent_same_direction,substantial_reversal,floor_label,response_floor,persistent_fraction_of_responders,recovered_fraction_of_responders,reversal_fraction_of_responders
0,cold,10802,10638,148,16,p90,0.272532,0.013701,0.984818,0.001481
1,cold_high_light,13578,12544,405,629,p90,0.272532,0.029828,0.923847,0.046325
2,drought,13347,11270,1934,143,p90,0.272532,0.144901,0.844385,0.010714
3,heat,2268,2124,118,26,p90,0.272532,0.052028,0.936508,0.011464
4,heat_drought,15092,14528,480,84,p90,0.272532,0.031805,0.962629,0.005566
5,heat_drought_high_light,10670,10194,418,58,p90,0.272532,0.039175,0.955389,0.005436
6,heat_high_light,1663,1509,130,24,p90,0.272532,0.078172,0.907396,0.014432
7,high_humidity,2721,2603,61,57,p90,0.272532,0.022418,0.956634,0.020948
8,high_light,3405,2233,1134,38,p90,0.272532,0.333040,0.655800,0.011160
9,cold,8915,8838,71,6,p95,0.342263,0.007964,0.991363,0.000673


In [25]:
persistent_count_sensitivity = (
    response_floor_sensitivity
    .pivot(
        index="condition",
        columns="floor_label",
        values="persistent_same_direction",
    )
    .reindex(condition_order)
)

persistent_count_sensitivity

floor_label,p90,p95,p99
condition,,,
cold,148,71,25
cold_high_light,405,187,44
drought,1934,1240,497
heat,118,70,25
heat_drought,480,344,129
heat_drought_high_light,418,255,78
heat_high_light,130,72,21
high_humidity,61,28,9
high_light,1134,779,361


In [26]:
persistent_fraction_sensitivity = (
    response_floor_sensitivity
    .pivot(
        index="condition",
        columns="floor_label",
        values="persistent_fraction_of_responders",
    )
    .reindex(condition_order)
)

persistent_fraction_sensitivity = (
    persistent_fraction_sensitivity * 100
)

persistent_fraction_sensitivity.round(2)

floor_label,p90,p95,p99
condition,,,
cold,1.37,0.80,0.60
cold_high_light,2.98,1.53,0.56
drought,14.49,10.38,5.96
heat,5.20,5.34,5.91
heat_drought,3.18,2.44,1.14
heat_drought_high_light,3.92,2.96,1.93
heat_high_light,7.82,8.85,11.48
high_humidity,2.24,1.64,1.49
high_light,33.30,39.95,62.24


In [27]:
recovered_fraction_sensitivity = (
    response_floor_sensitivity
    .pivot(
        index="condition",
        columns="floor_label",
        values="recovered_fraction_of_responders",
    )
    .reindex(condition_order)
)

recovered_fraction_sensitivity = (
    recovered_fraction_sensitivity * 100
)

recovered_fraction_sensitivity.round(2)

floor_label,p90,p95,p99
condition,,,
cold,98.48,99.14,99.40
cold_high_light,92.38,95.63,98.87
drought,84.44,88.87,93.78
heat,93.65,93.59,93.14
heat_drought,96.26,97.17,98.72
heat_drought_high_light,95.54,96.68,98.02
heat_high_light,90.74,89.56,88.52
high_humidity,95.66,97.02,97.52
high_light,65.58,59.18,37.24


In [28]:
from scipy.stats import spearmanr

In [29]:
rank_stability_rows = []

floor_pairs = [
    ("p90", "p95"),
    ("p95", "p99"),
    ("p90", "p99"),
]

for floor_a, floor_b in floor_pairs:

    rho, p_value = spearmanr(
        persistent_fraction_sensitivity[floor_a],
        persistent_fraction_sensitivity[floor_b],
    )

    rank_stability_rows.append(
        {
            "comparison": f"{floor_a}_vs_{floor_b}",
            "spearman_rho": rho,
            "p_value": p_value,
        }
    )

rank_stability = pd.DataFrame(
    rank_stability_rows
)

rank_stability

,comparison,spearman_rho,p_value
0,p90_vs_p95,0.983333,0.000002
1,p95_vs_p99,0.950000,0.000088
2,p90_vs_p99,0.900000,0.000943


In [30]:
persistent_fraction_ranks = (
    persistent_fraction_sensitivity
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

persistent_fraction_ranks

floor_label,p90,p95,p99
condition,,,
cold,9,9,8
cold_high_light,7,8,9
drought,2,2,3
heat,4,4,4
heat_drought,6,6,7
heat_drought_high_light,5,5,5
heat_high_light,3,3,2
high_humidity,8,7,6
high_light,1,1,1


In [31]:
persistent_sets_by_floor = {}

for floor_label, categories in (
    recovery_categories_by_floor.items()
):

    persistent_sets_by_floor[floor_label] = {}

    for condition in condition_order:

        persistent_sets_by_floor[
            floor_label
        ][condition] = set(
            categories.index[
                categories[condition]
                == "persistent_same_direction"
            ]
        )

In [32]:
nesting_rows = []

for condition in condition_order:

    p90_genes = persistent_sets_by_floor[
        "p90"
    ][condition]

    p95_genes = persistent_sets_by_floor[
        "p95"
    ][condition]

    p99_genes = persistent_sets_by_floor[
        "p99"
    ][condition]

    nesting_rows.append(
        {
            "condition": condition,
            "p95_subset_of_p90": (
                p95_genes.issubset(p90_genes)
            ),
            "p99_subset_of_p95": (
                p99_genes.issubset(p95_genes)
            ),
            "p99_subset_of_p90": (
                p99_genes.issubset(p90_genes)
            ),
        }
    )

persistent_set_nesting = pd.DataFrame(
    nesting_rows
)

persistent_set_nesting

,condition,p95_subset_of_p90,p99_subset_of_p95,p99_subset_of_p90
0,cold,True,True,True
1,cold_high_light,True,True,True
2,drought,True,True,True
3,heat,True,True,True
4,heat_drought,True,True,True
5,heat_drought_high_light,True,True,True
6,heat_high_light,True,True,True
7,high_humidity,True,True,True
8,high_light,True,True,True


In [33]:
gene_set_stability_rows = []

floor_pairs = [
    ("p90", "p95"),
    ("p95", "p99"),
    ("p90", "p99"),
]

for condition in condition_order:

    for floor_a, floor_b in floor_pairs:

        genes_a = persistent_sets_by_floor[
            floor_a
        ][condition]

        genes_b = persistent_sets_by_floor[
            floor_b
        ][condition]

        intersection = genes_a & genes_b
        union = genes_a | genes_b

        gene_set_stability_rows.append(
            {
                "condition": condition,
                "comparison": (
                    f"{floor_a}_vs_{floor_b}"
                ),
                "n_a": len(genes_a),
                "n_b": len(genes_b),
                "shared_genes": len(intersection),
                "jaccard": (
                    len(intersection) / len(union)
                    if len(union) > 0
                    else np.nan
                ),
            }
        )

gene_set_stability = pd.DataFrame(
    gene_set_stability_rows
)

gene_set_stability

,condition,comparison,n_a,n_b,shared_genes,jaccard
0,cold,p90_vs_p95,148,71,71,0.479730
1,cold,p95_vs_p99,71,25,25,0.352113
2,cold,p90_vs_p99,148,25,25,0.168919
3,cold_high_light,p90_vs_p95,405,187,187,0.461728
4,cold_high_light,p95_vs_p99,187,44,44,0.235294
5,cold_high_light,p90_vs_p99,405,44,44,0.108642
6,drought,p90_vs_p95,1934,1240,1240,0.641158
7,drought,p95_vs_p99,1240,497,497,0.400806
8,drought,p90_vs_p99,1934,497,497,0.256980
9,heat,p90_vs_p95,118,70,70,0.593220


In [34]:
gene_set_jaccard = (
    gene_set_stability
    .pivot(
        index="condition",
        columns="comparison",
        values="jaccard",
    )
)

gene_set_jaccard.round(3)

comparison,p90_vs_p95,p90_vs_p99,p95_vs_p99
condition,,,
cold,0.480,0.169,0.352
cold_high_light,0.462,0.109,0.235
drought,0.641,0.257,0.401
heat,0.593,0.212,0.357
heat_drought,0.717,0.269,0.375
heat_drought_high_light,0.610,0.187,0.306
heat_high_light,0.554,0.162,0.292
high_humidity,0.459,0.148,0.321
high_light,0.687,0.318,0.463


In [35]:
retention_rows = []

for condition in condition_order:

    p90 = persistent_sets_by_floor[
        "p90"
    ][condition]

    p95 = persistent_sets_by_floor[
        "p95"
    ][condition]

    p99 = persistent_sets_by_floor[
        "p99"
    ][condition]

    retention_rows.append(
        {
            "condition": condition,
            "p95_retained_from_p90": (
                len(p95) / len(p90)
                if len(p90) > 0
                else np.nan
            ),
            "p99_retained_from_p95": (
                len(p99) / len(p95)
                if len(p95) > 0
                else np.nan
            ),
            "p99_retained_from_p90": (
                len(p99) / len(p90)
                if len(p90) > 0
                else np.nan
            ),
        }
    )

persistent_gene_retention = pd.DataFrame(
    retention_rows
)

for column in [
    "p95_retained_from_p90",
    "p99_retained_from_p95",
    "p99_retained_from_p90",
]:
    persistent_gene_retention[column] *= 100

persistent_gene_retention.round(1)

,condition,p95_retained_from_p90,p99_retained_from_p95,p99_retained_from_p90
0,cold,48.0,35.2,16.9
1,cold_high_light,46.2,23.5,10.9
2,drought,64.1,40.1,25.7
3,heat,59.3,35.7,21.2
4,heat_drought,71.7,37.5,26.9
5,heat_drought_high_light,61.0,30.6,18.7
6,heat_high_light,55.4,29.2,16.2
7,high_humidity,45.9,32.1,14.8
8,high_light,68.7,46.3,31.8


In [36]:
persistence_breadth_by_floor = {}

for floor_label, categories in (
    recovery_categories_by_floor.items()
):

    persistent_matrix = (
        categories
        .eq("persistent_same_direction")
    )

    breadth = persistent_matrix.sum(axis=1)

    persistence_breadth_by_floor[
        floor_label
    ] = breadth

In [37]:
breadth_distribution_95 = (
    persistence_breadth_by_floor["p95"]
    .loc[
        persistence_breadth_by_floor["p95"] > 0
    ]
    .value_counts()
    .sort_index()
)

breadth_distribution_95

1    883
2    350
3    170
4    126
5     48
6     17
7     13
8      2
Name: count, dtype: int64

In [38]:
breadth_summary_rows = []

for floor_label, breadth in (
    persistence_breadth_by_floor.items()
):

    breadth_summary_rows.append(
        {
            "floor_label": floor_label,

            "persistent_at_least_1": (
                breadth >= 1
            ).sum(),

            "persistent_at_least_2": (
                breadth >= 2
            ).sum(),

            "persistent_at_least_3": (
                breadth >= 3
            ).sum(),

            "persistent_at_least_4": (
                breadth >= 4
            ).sum(),

            "persistent_at_least_5": (
                breadth >= 5
            ).sum(),

            "persistent_at_least_6": (
                breadth >= 6
            ).sum(),

            "maximum_breadth": breadth.max(),
        }
    )

breadth_sensitivity_summary = pd.DataFrame(
    breadth_summary_rows
)

breadth_sensitivity_summary

,floor_label,persistent_at_least_1,persistent_at_least_2,persistent_at_least_3,persistent_at_least_4,persistent_at_least_5,persistent_at_least_6,maximum_breadth
0,p90,2534,1124,576,344,149,63,9
1,p95,1609,726,376,206,80,32,8
2,p99,647,305,143,62,21,9,7


In [39]:
direction_robustness_rows = []

for floor_label, categories in (
    recovery_categories_by_floor.items()
):

    persistent_matrix = (
        categories
        .eq("persistent_same_direction")
    )

    breadth = persistent_matrix.sum(axis=1)

    persistent_direction = (
        np.sign(recovery_shift)
        .where(persistent_matrix)
    )

    direction_min = (
        persistent_direction.min(axis=1)
    )

    direction_max = (
        persistent_direction.max(axis=1)
    )

    multi_stress = breadth >= 2

    direction_consistent = (
        multi_stress
        & (direction_min == direction_max)
    )

    consistently_up = (
        direction_consistent
        & (direction_max == 1)
    )

    consistently_down = (
        direction_consistent
        & (direction_max == -1)
    )

    inconsistent = (
        multi_stress
        & ~direction_consistent
    )

    n_multi = int(multi_stress.sum())
    n_consistent = int(
        direction_consistent.sum()
    )
    n_up = int(
        consistently_up.sum()
    )
    n_down = int(
        consistently_down.sum()
    )
    n_inconsistent = int(
        inconsistent.sum()
    )

    direction_robustness_rows.append(
        {
            "floor_label": floor_label,
            "multi_stress_persistent_genes": n_multi,
            "directionally_consistent": n_consistent,
            "directionally_inconsistent": n_inconsistent,
            "consistent_up": n_up,
            "consistent_down": n_down,
            "percent_directionally_consistent": (
                100 * n_consistent / n_multi
                if n_multi > 0
                else np.nan
            ),
            "percent_consistent_genes_up": (
                100 * n_up / n_consistent
                if n_consistent > 0
                else np.nan
            ),
        }
    )

direction_robustness = pd.DataFrame(
    direction_robustness_rows
)

direction_robustness.round(2)

,floor_label,multi_stress_persistent_genes,directionally_consistent,directionally_inconsistent,consistent_up,consistent_down,percent_directionally_consistent,percent_consistent_genes_up
0,p90,1124,1102,22,1082,20,98.04,98.19
1,p95,726,720,6,711,9,99.17,98.75
2,p99,305,304,1,303,1,99.67,99.67


In [40]:
GO_SLIM_PATH = Path(
    "../data/external/arabidopsis_annotation/ATH_GO_GOSLIM.txt"
)

In [41]:
go_columns = [
    "gene_id",
    "tair_object_id",
    "object_name",
    "go_relationship",
    "go_term",
    "go_id",
    "go_numeric_id",
    "go_aspect",
    "go_slim_term",
    "evidence_code",
    "evidence_detail",
    "evidence_with",
    "reference",
    "annotator",
    "annotation_date",
]

In [42]:
go_annotations = pd.read_csv(
    GO_SLIM_PATH,
    sep="\t",
    header=None,
    names=go_columns,
    dtype=str,
    comment="!",
)

go_annotations.shape

(337091, 15)

In [43]:
go_slim_pairs = (
    go_annotations[
        [
            "gene_id",
            "go_aspect",
            "go_slim_term",
        ]
    ]
    .dropna(
        subset=[
            "gene_id",
            "go_aspect",
            "go_slim_term",
        ]
    )
    .drop_duplicates()
)

go_slim_pairs.shape

(188875, 3)

In [44]:
go_slim_pairs[
    [
        "go_aspect",
        "go_slim_term",
    ]
].drop_duplicates().shape

(98, 2)

In [45]:
high_breadth_sets = {}

for floor_label, breadth in (
    persistence_breadth_by_floor.items()
):

    high_breadth_sets[floor_label] = set(
        breadth.index[
            breadth >= 4
        ]
    )

In [46]:
{
    floor_label: len(genes)
    for floor_label, genes
    in high_breadth_sets.items()
}

{'p90': 344, 'p95': 206, 'p99': 62}

In [47]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

In [48]:
def run_go_slim_enrichment(
    foreground_ids,
    background_ids,
    go_slim_pairs,
):
    rows = []

    for aspect in ["P", "F", "C"]:

        aspect_pairs = (
            go_slim_pairs.loc[
                go_slim_pairs["go_aspect"] == aspect
            ]
        )

        annotatable_background = (
            set(aspect_pairs["gene_id"])
            & set(background_ids)
        )

        annotatable_foreground = (
            set(foreground_ids)
            & annotatable_background
        )

        background_total = len(
            annotatable_background
        )

        foreground_total = len(
            annotatable_foreground
        )

        terms = (
            aspect_pairs["go_slim_term"]
            .drop_duplicates()
        )

        for term in terms:

            term_genes = set(
                aspect_pairs.loc[
                    aspect_pairs["go_slim_term"]
                    == term,
                    "gene_id",
                ]
            )

            background_with_term = len(
                annotatable_background
                & term_genes
            )

            foreground_with_term = len(
                annotatable_foreground
                & term_genes
            )

            a = foreground_with_term

            b = (
                foreground_total
                - foreground_with_term
            )

            c = (
                background_with_term
                - foreground_with_term
            )

            d = (
                background_total
                - foreground_total
                - c
            )

            odds_ratio, p_value = (
                fisher_exact(
                    [
                        [a, b],
                        [c, d],
                    ],
                    alternative="greater",
                )
            )

            foreground_fraction = (
                a / foreground_total
                if foreground_total > 0
                else np.nan
            )

            background_fraction = (
                background_with_term
                / background_total
                if background_total > 0
                else np.nan
            )

            enrichment_ratio = (
                foreground_fraction
                / background_fraction
                if background_fraction > 0
                else np.nan
            )

            rows.append(
                {
                    "go_aspect": aspect,
                    "go_slim_term": term,
                    "foreground_with_term": a,
                    "foreground_total": foreground_total,
                    "background_with_term": (
                        background_with_term
                    ),
                    "background_total": (
                        background_total
                    ),
                    "enrichment_ratio": (
                        enrichment_ratio
                    ),
                    "odds_ratio": odds_ratio,
                    "p_value": p_value,
                }
            )

    enrichment = pd.DataFrame(rows)

    enrichment["fdr_bh"] = multipletests(
        enrichment["p_value"],
        method="fdr_bh",
    )[1]

    return enrichment

In [49]:
background_ids = set(
    informative_expression.index
)

In [50]:
go_slim_p95 = run_go_slim_enrichment(
    foreground_ids=high_breadth_sets["p95"],
    background_ids=background_ids,
    go_slim_pairs=go_slim_pairs,
)

In [51]:
go_slim_p95.shape

(98, 10)

In [52]:
go_slim_p95_significant = (
    go_slim_p95.loc[
        go_slim_p95["fdr_bh"] < 0.05
    ]
    .sort_values("fdr_bh")
)

len(go_slim_p95_significant)

24

In [53]:
go_slim_p95_significant[
    [
        "go_aspect",
        "go_slim_term",
        "foreground_with_term",
        "foreground_total",
        "background_with_term",
        "background_total",
        "enrichment_ratio",
        "fdr_bh",
    ]
].head(15)

,go_aspect,go_slim_term,foreground_with_term,foreground_total,background_with_term,background_total,enrichment_ratio,fdr_bh
16,P,response to biotic stimulus,74,168,1316,20524,6.869554,1.503207e-41
13,P,response to external stimulus,74,168,1410,20524,6.411584,9.063060e-40
15,P,response to stress,108,168,3678,20524,3.587276,5.092910e-39
25,P,response to chemical,91,168,2878,20524,3.862810,2.953618e-33
20,P,response to abiotic stimulus,59,168,1836,20524,3.925835,8.627229e-20
30,P,response to endogenous stimulus,41,168,1530,20524,3.273747,1.274456e-10
82,C,plasma membrane,52,206,3054,25866,2.137944,9.862632e-07
26,P,signal transduction,37,168,1822,20524,2.480882,2.150649e-06
39,P,cell death,10,168,220,20524,5.553030,1.529700e-04
86,C,extracellular region,42,206,2758,25866,1.912129,2.724112e-04


In [54]:
go_slim_enrichment_by_floor = {}
go_slim_significant_by_floor = {}

for floor_label in [
    "p90",
    "p95",
    "p99",
]:

    enrichment = run_go_slim_enrichment(
        foreground_ids=high_breadth_sets[
            floor_label
        ],
        background_ids=background_ids,
        go_slim_pairs=go_slim_pairs,
    )

    significant = (
        enrichment.loc[
            enrichment["fdr_bh"] < 0.05
        ]
        .sort_values("fdr_bh")
        .reset_index(drop=True)
    )

    go_slim_enrichment_by_floor[
        floor_label
    ] = enrichment

    go_slim_significant_by_floor[
        floor_label
    ] = significant

In [55]:
go_slim_significant_counts = pd.DataFrame(
    {
        "floor_label": [
            "p90",
            "p95",
            "p99",
        ],
        "high_breadth_genes": [
            len(
                high_breadth_sets[
                    "p90"
                ]
            ),
            len(
                high_breadth_sets[
                    "p95"
                ]
            ),
            len(
                high_breadth_sets[
                    "p99"
                ]
            ),
        ],
        "significant_go_slim_terms": [
            len(
                go_slim_significant_by_floor[
                    "p90"
                ]
            ),
            len(
                go_slim_significant_by_floor[
                    "p95"
                ]
            ),
            len(
                go_slim_significant_by_floor[
                    "p99"
                ]
            ),
        ],
    }
)

go_slim_significant_counts

,floor_label,high_breadth_genes,significant_go_slim_terms
0,p90,344,27
1,p95,206,24
2,p99,62,14


In [56]:
significant_term_sets = {}

for floor_label, df in (
    go_slim_significant_by_floor.items()
):

    significant_term_sets[
        floor_label
    ] = set(
        zip(
            df["go_aspect"],
            df["go_slim_term"],
        )
    )

In [57]:
{
    floor_label: len(terms)
    for floor_label, terms
    in significant_term_sets.items()
}

{'p90': 27, 'p95': 24, 'p99': 14}

In [58]:
functional_overlap_rows = []

floor_pairs = [
    ("p90", "p95"),
    ("p95", "p99"),
    ("p90", "p99"),
]

for floor_a, floor_b in floor_pairs:

    terms_a = significant_term_sets[
        floor_a
    ]

    terms_b = significant_term_sets[
        floor_b
    ]

    shared = terms_a & terms_b
    union = terms_a | terms_b

    functional_overlap_rows.append(
        {
            "comparison": (
                f"{floor_a}_vs_{floor_b}"
            ),
            "n_terms_a": len(terms_a),
            "n_terms_b": len(terms_b),
            "shared_significant_terms": (
                len(shared)
            ),
            "jaccard": (
                len(shared) / len(union)
                if len(union) > 0
                else np.nan
            ),
        }
    )

go_slim_functional_overlap = (
    pd.DataFrame(
        functional_overlap_rows
    )
)

go_slim_functional_overlap.round(3)

,comparison,n_terms_a,n_terms_b,shared_significant_terms,jaccard
0,p90_vs_p95,27,24,22,0.759
1,p95_vs_p99,24,14,13,0.520
2,p90_vs_p99,27,14,13,0.464


In [59]:
primary_terms = (
    go_slim_significant_by_floor[
        "p95"
    ][
        [
            "go_aspect",
            "go_slim_term",
        ]
    ]
    .copy()
)

term_sensitivity_tables = []

for floor_label in [
    "p90",
    "p95",
    "p99",
]:

    df = (
        go_slim_enrichment_by_floor[
            floor_label
        ][
            [
                "go_aspect",
                "go_slim_term",
                "foreground_with_term",
                "foreground_total",
                "enrichment_ratio",
                "fdr_bh",
            ]
        ]
        .copy()
    )

    df = df.rename(
        columns={
            "foreground_with_term":
                f"{floor_label}_foreground_with_term",
            "foreground_total":
                f"{floor_label}_foreground_total",
            "enrichment_ratio":
                f"{floor_label}_enrichment_ratio",
            "fdr_bh":
                f"{floor_label}_fdr_bh",
        }
    )

    term_sensitivity_tables.append(
        df
    )

In [60]:
primary_term_sensitivity = (
    primary_terms
)

for df in term_sensitivity_tables:

    primary_term_sensitivity = (
        primary_term_sensitivity.merge(
            df,
            on=[
                "go_aspect",
                "go_slim_term",
            ],
            how="left",
            validate="one_to_one",
        )
    )

In [61]:
for floor_label in [
    "p90",
    "p95",
    "p99",
]:

    primary_term_sensitivity[
        f"{floor_label}_significant"
    ] = (
        primary_term_sensitivity[
            f"{floor_label}_fdr_bh"
        ] < 0.05
    )

In [62]:
primary_term_sensitivity[
    [
        "go_aspect",
        "go_slim_term",
        "p90_enrichment_ratio",
        "p90_fdr_bh",
        "p90_significant",
        "p95_enrichment_ratio",
        "p95_fdr_bh",
        "p95_significant",
        "p99_enrichment_ratio",
        "p99_fdr_bh",
        "p99_significant",
    ]
].round(4)

,go_aspect,go_slim_term,p90_enrichment_ratio,p90_fdr_bh,p90_significant,p95_enrichment_ratio,p95_fdr_bh,p95_significant,p99_enrichment_ratio,p99_fdr_bh,p99_significant
0,P,response to biotic stimulus,6.4084,0.0000,True,6.8696,0.0000,True,9.3574,0.0000,True
1,P,response to external stimulus,5.9812,0.0000,True,6.4116,0.0000,True,8.7336,0.0000,True
2,P,response to stress,3.3278,0.0000,True,3.5873,0.0000,True,3.9061,0.0000,True
3,P,response to chemical,3.3712,0.0000,True,3.8628,0.0000,True,4.2788,0.0000,True
4,P,response to abiotic stimulus,3.2926,0.0000,True,3.9258,0.0000,True,3.3536,0.0004,True
5,P,response to endogenous stimulus,2.7804,0.0000,True,3.2737,0.0000,True,2.6829,0.0255,True
6,C,plasma membrane,1.9943,0.0000,True,2.1379,0.0000,True,2.3223,0.0062,True
7,P,signal transduction,2.4987,0.0000,True,2.4809,0.0000,True,2.7035,0.0094,True
8,P,cell death,5.7671,0.0000,True,5.5530,0.0002,True,5.5975,0.0948,False
9,C,extracellular region,1.9084,0.0000,True,1.9121,0.0003,True,2.4203,0.0062,True


In [63]:
primary_term_robustness_summary = {
    "primary_p95_terms": len(
        primary_term_sensitivity
    ),

    "enriched_above_1_at_all_floors": (
        (
            primary_term_sensitivity[
                [
                    "p90_enrichment_ratio",
                    "p95_enrichment_ratio",
                    "p99_enrichment_ratio",
                ]
            ] > 1
        )
        .all(axis=1)
        .sum()
    ),

    "enriched_at_least_2x_at_all_floors": (
        (
            primary_term_sensitivity[
                [
                    "p90_enrichment_ratio",
                    "p95_enrichment_ratio",
                    "p99_enrichment_ratio",
                ]
            ] >= 2
        )
        .all(axis=1)
        .sum()
    ),

    "significant_at_all_floors": (
        primary_term_sensitivity[
            [
                "p90_significant",
                "p95_significant",
                "p99_significant",
            ]
        ]
        .all(axis=1)
        .sum()
    ),
}

primary_term_robustness_summary

{'primary_p95_terms': 24,
 'enriched_above_1_at_all_floors': np.int64(23),
 'enriched_at_least_2x_at_all_floors': np.int64(12),
 'significant_at_all_floors': np.int64(12)}

In [64]:
# Condition-level response-floor sensitivity
response_floor_sensitivity.to_csv(
    ROBUSTNESS_DIR / "response_floor_sensitivity.csv",
    index=False,
)

persistent_count_sensitivity.reset_index().to_csv(
    ROBUSTNESS_DIR / "persistent_count_sensitivity.csv",
    index=False,
)

persistent_fraction_sensitivity.reset_index().to_csv(
    ROBUSTNESS_DIR / "persistent_fraction_sensitivity.csv",
    index=False,
)

recovered_fraction_sensitivity.reset_index().to_csv(
    ROBUSTNESS_DIR / "recovered_fraction_sensitivity.csv",
    index=False,
)

# Ranking and gene-set stability
rank_stability.to_csv(
    ROBUSTNESS_DIR / "persistent_fraction_rank_stability.csv",
    index=False,
)

persistent_fraction_ranks.reset_index().to_csv(
    ROBUSTNESS_DIR / "persistent_fraction_ranks.csv",
    index=False,
)

persistent_set_nesting.to_csv(
    ROBUSTNESS_DIR / "persistent_set_nesting.csv",
    index=False,
)

persistent_gene_retention.to_csv(
    ROBUSTNESS_DIR / "persistent_gene_retention.csv",
    index=False,
)

# Persistence breadth and direction
breadth_sensitivity_summary.to_csv(
    ROBUSTNESS_DIR / "persistence_breadth_sensitivity.csv",
    index=False,
)

direction_robustness.to_csv(
    ROBUSTNESS_DIR / "directional_consistency_sensitivity.csv",
    index=False,
)

# Functional robustness
go_slim_significant_counts.to_csv(
    ROBUSTNESS_DIR / "go_slim_significant_counts_by_floor.csv",
    index=False,
)

go_slim_functional_overlap.to_csv(
    ROBUSTNESS_DIR / "go_slim_significant_term_overlap.csv",
    index=False,
)

primary_term_sensitivity.to_csv(
    ROBUSTNESS_DIR / "primary_go_slim_term_sensitivity.csv",
    index=False,
)

In [65]:
for floor_label, enrichment in (
    go_slim_enrichment_by_floor.items()
):

    enrichment.to_csv(
        ROBUSTNESS_DIR
        / f"go_slim_enrichment_{floor_label}_all.csv",
        index=False,
    )

In [66]:
for floor_label, significant in (
    go_slim_significant_by_floor.items()
):

    significant.to_csv(
        ROBUSTNESS_DIR
        / f"go_slim_enrichment_{floor_label}_significant.csv",
        index=False,
    )

In [67]:
sorted(
    path.name
    for path in ROBUSTNESS_DIR.iterdir()
    if path.is_file()
)

['directional_consistency_sensitivity.csv',
 'go_slim_enrichment_p90_all.csv',
 'go_slim_enrichment_p90_significant.csv',
 'go_slim_enrichment_p95_all.csv',
 'go_slim_enrichment_p95_significant.csv',
 'go_slim_enrichment_p99_all.csv',
 'go_slim_enrichment_p99_significant.csv',
 'go_slim_significant_counts_by_floor.csv',
 'go_slim_significant_term_overlap.csv',
 'persistence_breadth_sensitivity.csv',
 'persistent_count_sensitivity.csv',
 'persistent_fraction_rank_stability.csv',
 'persistent_fraction_ranks.csv',
 'persistent_fraction_sensitivity.csv',
 'persistent_gene_retention.csv',
 'persistent_set_nesting.csv',
 'primary_go_slim_term_sensitivity.csv',
 'recovered_fraction_sensitivity.csv',
 'response_floor_sensitivity.csv']